In [1]:
# ===== Task 3: Try other machine learning models =====
# The only hard rule here is no deep learning and no LLMs.

# What the task is asking for:
# 3a, code for all the models we tried, with comments saying what they were and
#     what the key hyperparameters were. so I am keeping the models that lost as
#     well, not just the winner, because the task explicitly asks for all of them.
# 3b, submit predicted labels to kaggle under the registered team name.

# The plan I ended up with after testing things rather than guessing:
#   1. get a baseline with a few standard models on the given 5000 features
#   2. build my own features from the raw text instead, and see if that beats it
#   3. try gradient boosting on top of the better features
#   4. check whether the train and test sets are even the same kind of data
# step 4 turned out to be the most important thing in the whole task.

In [2]:
# Load everything we need.
# task 1 and 2 both worked from train_features.csv, which is the ready made
# tf-idf the competition gives us. this time I also load train.csv and test.csv
# because those have the actual raw text in them, and I want to try building my
# own features later on.
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score
from scipy.sparse import hstack
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

DATA = "50-007-machine-learning-may-2026/"

# the given tf-idf features, same as task 1 and 2 used
feat_df = pd.read_csv(DATA + "train_features.csv")
X_given = feat_df.drop(columns=["id", "label"]).values
y = feat_df["label"].values

# the raw text
train_raw = pd.read_csv(DATA + "train.csv")
test_raw = pd.read_csv(DATA + "test.csv")
train_raw["text"] = train_raw["text"].fillna("")
test_raw["text"] = test_raw["text"].fillna("")

print("given features", X_given.shape)
print("raw train     ", train_raw.shape)
print("raw test      ", test_raw.shape)

given features (20000, 5000)
raw train      (20000, 3)
raw test       (6999, 2)


In [3]:
# Split once and reuse the same split everywhere, so that every number below is
# comparable. stratify keeps the class balance the same in both halves and the
# fixed seed means I am not chasing noise between runs.
# note the raw text split uses the same seed, so row for row it is the same
# documents in the same halves as the feature split. that matters, otherwise I
# would be comparing models on different data and not on different features.
Xa, Xb, ya, yb = train_test_split(X_given, y, test_size=0.2, stratify=y, random_state=42)
ta, tb, ya_t, yb_t = train_test_split(train_raw["text"], train_raw["label"].values,
                                      test_size=0.2, stratify=train_raw["label"], random_state=42)
assert (ya == ya_t).all(), "the two splits must line up or the comparison is meaningless"

print("train", Xa.shape[0], "rows | holdout", Xb.shape[0], "rows")


# macro f1 is the competition metric, same as task 1 and 2. sklearn is allowed
# in this task so I am not hand rolling it this time.
def macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")


# Most of these models give a score rather than a hard 0 or 1, and where we put
# the cut off changes the macro f1 quite a bit. task 1 already showed that the
# default 0.5 is usually not the best place for it, so I sweep the cut off and
# report the best one. I sweep over percentiles of the scores rather than fixed
# numbers, because a decision_function from an svm is not on the same scale as a
# probability from logistic regression and fixed values would not work for both.
def tuned_macro(scores, y_true):
    best_t, best_m = 0.0, -1.0
    for t in np.percentile(scores, np.arange(5, 96, 1)):
        m = macro(y_true, (scores >= t).astype(int))
        if m > best_m:
            best_t, best_m = t, m
    return best_t, best_m

train 16000 rows | holdout 4000 rows


In [4]:
# Attempt 1, standard models on the 5000 features we were given.
# these are the obvious things to reach for on tf-idf text. logistic regression
# and linear svm because linear models are the usual first choice for text, and
# complement naive bayes because it is built for exactly this shape of data and
# costs nothing to try.
# hyperparameters are on each line. where I tried more than one value of C I have
# left both in, since the task wants to see what was tried.
print("model                          default    tuned")
for name, make, score in [
    ("LogisticRegression C=1",  lambda: LogisticRegression(max_iter=2000),        lambda m, X: m.predict_proba(X)[:, 1]),
    ("LogisticRegression C=10", lambda: LogisticRegression(C=10, max_iter=2000),  lambda m, X: m.predict_proba(X)[:, 1]),
    ("LinearSVC C=1",           lambda: LinearSVC(C=1),                            lambda m, X: m.decision_function(X)),
    ("ComplementNB",            lambda: ComplementNB(),                            lambda m, X: m.predict_proba(X)[:, 1]),
]:
    m = make().fit(Xa, ya)
    print("%-28s  %.4f    %.4f" % (name, macro(yb, m.predict(Xb)), tuned_macro(score(m, Xb), yb)[1]))

# Everything lands in the same place, around 0.73 to 0.74. our own hand written
# logistic regression from task 1 got 0.7431, so swapping in sklearn models and
# tuning them buys us essentially nothing.
# that is worth stopping on rather than trying yet another model. when four
# different algorithms all plateau at the same number, the thing holding us back
# is not the algorithm, it is the features. so the next thing to try is better
# features and not a better model.

model                          default    tuned
LogisticRegression C=1        0.7058    0.7424
LogisticRegression C=10       0.7312    0.7378
LinearSVC C=1                 0.7275    0.7352
ComplementNB                  0.6483    0.6794


In [5]:
# Attempt 2, build my own features from the raw text.
# the 5000 features we were given are a summary of the text that somebody else
# chose. the actual documents are sitting in train.csv, so nothing stops me
# making my own and seeing if I can do better than their summary.
#
# two vectorisers, because they pick up different things:
#   word 1-2 grams catch which words and short phrases get used
#   char_wb 3-5 grams catch spelling, punctuation and word endings, so they see
#   style rather than topic. that matters for telling machine text from human
#   text, because a lot of the giveaway is in how something is written rather
#   than what it is about.
# sublinear_tf uses 1+log(tf) instead of raw counts, so a word appearing twenty
# times does not count twenty times as much as one appearing once.
# min_df=2 throws away anything appearing in only one document, which is mostly
# typos and would just be noise.
#
# important, fit_transform on train and transform on test. same leakage rule as
# the pca in task 2. if I fit the vectoriser on the test text as well then the
# vocabulary has seen the test set and the score is not honest any more.
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50000,  sublinear_tf=True, min_df=2)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=100000, sublinear_tf=True, min_df=2)

Wa, Wb = word_vec.fit_transform(ta), word_vec.transform(tb)
Ca, Cb = char_vec.fit_transform(ta), char_vec.transform(tb)

# hstack glues the two feature blocks side by side into one matrix. these stay
# sparse the whole way, which is why 150000 columns is not a memory problem.
Fa = hstack([Wa, Ca]).tocsr()
Fb = hstack([Wb, Cb]).tocsr()
print("my features", Fa.shape, "vs the given", Xa.shape)

print()
print("features                          tuned macro f1")
for name, A, B in [("word 1-2gram only", Wa, Wb), ("char_wb 3-5gram only", Ca, Cb), ("word + char together", Fa, Fb)]:
    m = LinearSVC(C=1).fit(A, ya)
    print("%-32s  %.4f" % (name, tuned_macro(m.decision_function(B), yb)[1]))

# Both feature families on their own already beat everything from attempt 1, and
# putting them together beats either one alone. that last bit is the useful part,
# if they were finding the same signal then combining them would not help, so the
# fact that it does means they really are picking up different things.

my features (16000, 150000) vs the given (16000, 5000)

features                          tuned macro f1
word 1-2gram only                 0.7891
char_wb 3-5gram only              0.7954
word + char together              0.8078


In [ ]:
# Attempt 3, gradient boosting.
# up to here everything has been a linear model. boosting works completely
# differently, it builds a lot of small decision trees where each one tries to
# fix the mistakes of the ones before it, so it can pick up combinations of
# features that a straight line cannot.
# I try it on both feature sets, because I want to know whether boosting helps on
# its own or only once the features are good.
#
# lightgbm and xgboost do the same kind of thing. I am running both because the
# task asks for everything tried, and lightgbm handles very wide sparse data
# better which matters once we are at 150000 columns.
print("=== on the given 5000 features ===")
m = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, tree_method="hist", n_jobs=-1).fit(Xa, ya)
print("XGBoost   n_estimators=400 max_depth=6 lr=0.1    %.4f" % tuned_macro(m.predict_proba(Xb)[:, 1], yb)[1])
m = LGBMClassifier(n_estimators=400, num_leaves=63, learning_rate=0.1, n_jobs=-1, verbose=-1).fit(Xa, ya)
print("LightGBM  n_estimators=400 num_leaves=63 lr=0.1  %.4f" % tuned_macro(m.predict_proba(Xb)[:, 1], yb)[1])

print()
print("=== on my own word+char features (this takes a while) ===")
t = time.time()
lgb = LGBMClassifier(n_estimators=500, num_leaves=63, learning_rate=0.1, n_jobs=-1, verbose=-1).fit(Fa, ya)
s_lgb = lgb.predict_proba(Fb)[:, 1]
print("LightGBM  n_estimators=500 num_leaves=63 lr=0.1  %.4f  (%.0fs)" % (tuned_macro(s_lgb, yb)[1], time.time() - t))

svc = LinearSVC(C=1).fit(Fa, ya)
s_svc = svc.decision_function(Fb)
print("LinearSVC C=1                                    %.4f" % tuned_macro(s_svc, yb)[1])

# On the given features boosting is no better than the linear models, all still
# stuck around 0.74. on my own features it is clearly better than the svm.
# so boosting only pays off once the features are worth something, which is a
# reasonable thing to expect. trees need features that actually carry signal
# before they have anything useful to split on.
# worth noting lightgbm takes about 170 seconds against the svm's 1 second, so
# while I am trying things out I use the svm and only run lightgbm on a real
# candidate.

=== on the given 5000 features ===
XGBoost   n_estimators=400 max_depth=6 lr=0.1    0.7387
LightGBM  n_estimators=400 num_leaves=63 lr=0.1  0.7473

=== on my own word+char features (this one takes about 3 minutes) ===
LightGBM  n_estimators=500 num_leaves=63 lr=0.1  0.8389  (259s)
LinearSVC C=1                                    0.8078


In [7]:
# Attempt 4, blend the two best models.
# the svm and the boosting model are built on completely different ideas, so they
# tend to get different documents wrong. when that is true, averaging the two
# usually lands somewhere better than either on its own, because the mistakes do
# not line up.
# they are not on the same scale though, the svm gives a signed distance and
# lightgbm gives a probability between 0 and 1. so I standardise both to mean 0
# and standard deviation 1 first, otherwise whichever one has bigger numbers
# would dominate the average for no good reason.
def z(x):
    return (x - x.mean()) / x.std()

print("blend weight (lightgbm share)   tuned macro f1")
for w in [0.0, 0.3, 0.5, 0.7, 1.0]:
    print("   %.0f%%                          %.4f" % (w * 100, tuned_macro(w * z(s_lgb) + (1 - w) * z(s_svc), yb)[1]))

# 50/50 comes out best but only just, it is worth about 0.002 over lightgbm on
# its own. that is small enough that I would not trust it too far on a single
# holdout, but it does not cost anything either so I keep it.

blend weight (lightgbm share)   tuned macro f1
   0%                          0.8078
   30%                          0.8386
   50%                          0.8413
   70%                          0.8408
   100%                          0.8389


In [8]:
# Attempt 5, and this is the one that actually mattered.
# every number above is measured on a holdout taken out of the training set. that
# only tells me how well I do on data that looks like the training data. before
# trusting any of it I wanted to check whether the test set even looks like the
# training set.
#
# the check is called adversarial validation. I throw away the real labels, label
# every training document 0 and every test document 1, and try to train a model
# to tell them apart. if the two sets come from the same place then no model can
# do it and the AUC sits around 0.5. if the AUC is high then they are different
# data and my holdout scores are measuring the wrong thing.
#
# I noticed earlier that the ids are not all the same format. train is entirely
# uuid style, but test is a mix of uuid style and plain integers, so I check the
# two test groups separately as well.
from sklearn.model_selection import cross_val_score

test_raw["is_uuid"] = test_raw["id"].astype(str).str.contains("-")

all_text = pd.concat([train_raw["text"], test_raw["text"]])
adv_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True, min_df=2)
Vall = adv_vec.fit_transform(all_text)
is_test = np.r_[np.zeros(len(train_raw)), np.ones(len(test_raw))]

print("train vs all of test      AUC %.4f" % cross_val_score(LinearSVC(C=1), Vall, is_test, cv=3, scoring="roc_auc").mean())
for group, name in [(True, "uuid-id test docs"), (False, "int-id test docs ")]:
    keep = np.r_[np.ones(len(train_raw), bool), (test_raw["is_uuid"] == group).values]
    lab = np.r_[np.zeros(len(train_raw)), np.ones((test_raw["is_uuid"] == group).sum())]
    print("train vs %s AUC %.4f" % (name, cross_val_score(LinearSVC(C=1), Vall[keep], lab, cv=3, scoring="roc_auc").mean()))

# The uuid ones come out at about 0.49, which is the same as guessing, so those
# really are the same kind of data as the training set.
# the integer ones come out at about 0.95, which means a model can tell them
# apart almost perfectly. they are not the same kind of data at all.
# and there are 5000 of them out of 6999, so 71% of what we are graded on is
# something our training set barely contains.

train vs all of test      AUC 0.8016
train vs uuid-id test docs AUC 0.4875
train vs int-id test docs  AUC 0.9504


In [9]:
# So what is different about them? worth looking rather than guessing.
print("median document length in characters")
print("  train            %.0f" % train_raw["text"].str.len().median())
print("  uuid-id test     %.0f" % test_raw.loc[test_raw["is_uuid"], "text"].str.len().median())
print("  int-id test      %.0f" % test_raw.loc[~test_raw["is_uuid"], "text"].str.len().median())

print()
print("how often certain words turn up")
print("group        strengths  weaknesses  the paper   reviewer")
for name, s in [("train    ", train_raw["text"]),
                ("test uuid", test_raw.loc[test_raw["is_uuid"], "text"]),
                ("test int ", test_raw.loc[~test_raw["is_uuid"], "text"])]:
    low = s.str.lower()
    print("%s      %.3f      %.3f      %.3f      %.3f" % (
        name, low.str.contains("strength").mean(), low.str.contains("weakness").mean(),
        low.str.contains("the paper").mean(), low.str.contains("reviewer").mean()))

print()
print("an example int-id test document:")
print(" ", test_raw.loc[~test_raw["is_uuid"], "text"].iloc[0][:300].replace("\n", " "))

# The integer id documents are academic peer reviews. reviewer shows up in about
# 16% of them against 0.2% of the training set, and they are around 60% longer.
# our training data is arxiv abstracts, news snippets and bits of game text, so
# there are basically no peer reviews in it at all.
#
# this explains the leaderboard. everyone including us is scoring higher on their
# own holdout than they are on kaggle, because the holdout is the 29% of the test
# set that looks like training data and the leaderboard is mostly the other 71%.
#
# it also changes which features I should trust. content words that separate
# human from machine in an abstract will not carry over to a peer review, the
# vocabulary is completely different. what does carry over is style, punctuation
# and sentence rhythm and word endings, which is what the char n-grams pick up.
# that is an argument for leaning on the char features, and something the
# leaderboard can confirm or deny.

median document length in characters
  train            1146
  uuid-id test     1189
  int-id test      1862

how often certain words turn up
group        strengths  weaknesses  the paper   reviewer
train          0.042      0.018      0.032      0.002
test uuid      0.039      0.014      0.026      0.002
test int       0.192      0.142      0.233      0.159

an example int-id test document:
  The paper presents an analytical derivation of the Predictive Normalized Maximum Likelihood (pNML) regret for single-layer neural networks, which is then applied to pre-trained deep neural networks for out-of-distribution (OOD) detection. The core idea is to use the pNML regret as a measure to disti


In [ ]:
# Attempt 6, pseudo labelling. keeping this in because the task wants everything
# I tried, but I did not use it in the end
#
# the idea, since none of the training data is peer reviews, is to make some. run
# the model on the test set, take the documents it is most confident about, treat
# its own guesses as if they were real labels, add them to the training data and
# retrain. it is a standard semi supervised trick and it uses no test labels, so
# it is allowed, and it is nowhere near deep learning.
Fa_test = hstack([word_vec.transform(test_raw["text"]), char_vec.transform(test_raw["text"])]).tocsr()
d_test = svc.decision_function(Fa_test)

print("how confident the model is on each test group, mean absolute score")
print("  uuid-id  %.3f" % np.abs(d_test[test_raw["is_uuid"].values]).mean())
print("  int-id   %.3f" % np.abs(d_test[~test_raw["is_uuid"].values]).mean())

from scipy.sparse import vstack
base = tuned_macro(s_svc, yb)[1]
print()
print("adding confident int-id docs back into training:")
for q in [50, 70, 90]:
    cut = np.percentile(np.abs(d_test[~test_raw["is_uuid"].values]), 100 - q)
    sel = (~test_raw["is_uuid"].values) & (np.abs(d_test) >= cut)
    m = LinearSVC(C=1).fit(vstack([Fa, Fa_test[sel]]).tocsr(),
                           np.r_[ya, (d_test[sel] > 0).astype(int)])
    print("  top %2d%% confident, %4d docs added   holdout %.4f  (baseline %.4f)" % (
        q, sel.sum(), tuned_macro(m.decision_function(Fb), yb)[1], base))

# It comes out flat, and worse when I add more. but the problem is that
# this holdout cannot measure the thing pseudo labelling is meant to fix. the
# holdout is all in distribution data and the whole point of the trick is to help
# on the out of distribution 71%, which has no labels for me to check against.
# so this is untested rather than disproven. the only way to know would be to
# spend a kaggle submission on it, and I would rather spend those on the model I
# have actual evidence for.

how confident the model is on each test group, mean absolute score
  uuid-id  0.782
  int-id   0.730

adding confident int-id docs back into training:
  top 50% confident, 2500 docs added   holdout 0.8094  (baseline 0.8078)
  top 70% confident, 3500 docs added   holdout 0.8034  (baseline 0.8078)
  top 90% confident, 4500 docs added   holdout 0.8034  (baseline 0.8078)


In [11]:
# Attempt 7, testing what the literature actually says.
# I read a paper saying svm models do best on this kind of problem, because they
# cope well with lots of features and generalise well to samples that are not
# quite like what they were trained on.
# the first half of that we have already seen. LinearSVC is an svm and it beat
# everything else on the given 5000 features, so that part checks out.
# the second half is the interesting claim given what attempt 5 turned up. if
# svms really do hold up better on data that is not like the training set, then
# the svm might beat lightgbm on the actual leaderboard even though it loses on
# my holdout, because my holdout is the 29% that does look like training data.
#
# I cannot test that directly, the shifted 71% has no labels. so I fake a shift
# using the training data instead. I use the adversarial model from attempt 5 to
# score every training document by how test-like it is, train on the 10000 least
# test-like ones and score the 10000 most test-like ones. that is a smaller
# version of the real problem, fit on one kind of data and get marked on another.
# I run the same models on a plain random split too, otherwise I have nothing to
# compare the drop against.
#
# this cell takes about 6 minutes, most of it the two lightgbm fits.
from sklearn.linear_model import SGDClassifier

adv_model = LogisticRegression(max_iter=1000).fit(Vall, is_test)
test_like = adv_model.predict_proba(Vall[:len(train_raw)])[:, 1]

order = np.argsort(test_like)
lo, hi = order[:10000], order[10000:]        # least test-like, most test-like
print("mean test-likeness   train half %.3f   eval half %.3f" % (test_like[lo].mean(), test_like[hi].mean()))

def build(idx_fit, idx_eval):
    # fit fresh vectorisers on whichever half is acting as training data, same
    # leakage rule as always
    w = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True, min_df=2)
    c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=100000, sublinear_tf=True, min_df=2)
    A = hstack([w.fit_transform(train_raw["text"].iloc[idx_fit]), c.fit_transform(train_raw["text"].iloc[idx_fit])]).tocsr()
    B = hstack([w.transform(train_raw["text"].iloc[idx_eval]),    c.transform(train_raw["text"].iloc[idx_eval])]).tocsr()
    return A, B, train_raw["label"].values[idx_fit], train_raw["label"].values[idx_eval]

Sa, Sb, sya, syb = build(lo, hi)                                  # the shifted split
rng = np.random.default_rng(0); perm = rng.permutation(len(train_raw))
Ra, Rb, rya, ryb = build(perm[:10000], perm[10000:])              # a plain random split as the control

print()
print("%-30s %-14s %-14s %s" % ("model", "random split", "shifted split", "drop"))
for name, make in [
    ("LinearSVC C=0.1",          lambda: LinearSVC(C=0.1)),
    ("LinearSVC C=1",            lambda: LinearSVC(C=1)),
    ("LinearSVC C=10",           lambda: LinearSVC(C=10)),
    ("LogisticRegression C=10",  lambda: LogisticRegression(C=10, max_iter=2000)),
    ("SGD hinge, a linear svm",  lambda: SGDClassifier(loss="hinge", alpha=1e-5, max_iter=30, random_state=0)),
    ("LightGBM 500 x 63",        lambda: LGBMClassifier(n_estimators=500, num_leaves=63, learning_rate=0.1, n_jobs=-1, verbose=-1)),
]:
    s = lambda m, X: m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)
    a = tuned_macro(s(make().fit(Ra, rya), Rb), ryb)[1]
    b = tuned_macro(s(make().fit(Sa, sya), Sb), syb)[1]
    print("%-30s %.4f         %.4f         %+.4f" % (name, a, b, b - a))

# The result does not back the claim, at least not here. every model loses about
# the same amount when the split is shifted, roughly 0.02, and lightgbm's drop is
# right in the middle rather than being the worst. it also stays about 0.03 ahead
# of the svms even on the shifted split.
# C makes almost no difference either, 0.1 and 1 and 10 all land within 0.01, so
# there is no hidden win from tuning the svm harder.
#
# the honest limit of this test is that my fake shift is much gentler than the
# real one. test-likeness goes from 0.095 to 0.206 between my two halves, while
# the real gap between the training set and the peer reviews was an AUC of 0.95.
# so this shows svms have no advantage under a mild shift. it cannot rule out
# that they hold up better under the severe one, and the only way to settle that
# is to spend a submission and look at the leaderboard.
#
# so I keep lightgbm, but I keep the svm in the blend as well rather than going
# pure lightgbm. half the decision still comes from the model the literature
# recommends, which hedges against exactly the case this test could not measure.

mean test-likeness   train half 0.095   eval half 0.206

model                          random split   shifted split  drop
LinearSVC C=0.1                0.7943         0.7728         -0.0215
LinearSVC C=1                  0.7946         0.7721         -0.0225
LinearSVC C=10                 0.7855         0.7649         -0.0206
LogisticRegression C=10        0.7965         0.7709         -0.0256
SGD hinge, a linear svm        0.7853         0.7667         -0.0186
LightGBM 500 x 63              0.8286         0.8056         -0.0230


In [12]:
# Final model and the submission file.
# picking the 50/50 blend of lightgbm and the linear svm on my own word and char
# features, because that is the best number I have evidence for.
# refit both on all 20000 training rows now, not just the 16000 from the split.
# the split was there to compare options, and now that the choice is made there is
# no reason to throw away a fifth of the data.
print("refitting the vectorisers on all the training text")
word_full = TfidfVectorizer(ngram_range=(1, 2), max_features=50000,  sublinear_tf=True, min_df=2)
char_full = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=100000, sublinear_tf=True, min_df=2)
F_train = hstack([word_full.fit_transform(train_raw["text"]), char_full.fit_transform(train_raw["text"])]).tocsr()
F_test  = hstack([word_full.transform(test_raw["text"]),      char_full.transform(test_raw["text"])]).tocsr()
y_full = train_raw["label"].values
print("train", F_train.shape, " test", F_test.shape)

t = time.time()
svc_f = LinearSVC(C=1).fit(F_train, y_full)
lgb_f = LGBMClassifier(n_estimators=500, num_leaves=63, learning_rate=0.1, n_jobs=-1, verbose=-1).fit(F_train, y_full)
print("fitted both in %.0fs" % (time.time() - t))

blend = 0.5 * z(lgb_f.predict_proba(F_test)[:, 1]) + 0.5 * z(svc_f.decision_function(F_test))

# The cut off is the last decision. I cannot tune it on the test set because there
# are no labels, so I take the threshold that the same blend wanted on the holdout
# and apply the equivalent percentile here. going by the class balance instead
# would assume the test set has the same 62.5% split as training, and after what
# attempt 5 turned up I would not assume that.
best_t, best_m = tuned_macro(0.5 * z(s_lgb) + 0.5 * z(s_svc), yb)
pct = (( (0.5 * z(s_lgb) + 0.5 * z(s_svc)) < best_t).mean()) * 100
cut = np.percentile(blend, pct)
print("holdout wanted the cut at the %.0fth percentile, macro f1 %.4f there" % (pct, best_m))

pred = (blend >= cut).astype(int)
sub = pd.DataFrame({"id": test_raw["id"].values, "label": pred})
sub.to_csv("Task3_Prediction.csv", index=False)
print("wrote Task3_Prediction.csv", sub.shape)
print("predicted class 1 rate %.4f  (train rate %.4f)" % (pred.mean(), y_full.mean()))

refitting the vectorisers on all the training text
train (20000, 150000)  test (6999, 150000)
fitted both in 255s
holdout wanted the cut at the 34th percentile, macro f1 0.8413 there
wrote Task3_Prediction.csv (6999, 2)
predicted class 1 rate 0.6600  (train rate 0.6252)


In [13]:
# Second submission file, the svm on its own.
# attempt 7 could not settle whether the svm holds up better than boosting on the
# genuinely shifted part of the test set, because nothing local can measure that.
# the leaderboard can. so I also write out a pure LinearSVC version, and if I can
# spare a submission I upload both and compare. if the svm scores closer to the
# blend than my holdout says it should, that is real evidence the literature was
# right about generalisation and my local numbers were flattering the boosting.
# svc_f is already fitted on all 20000 rows from the cell above, so this is just
# a different way of turning it into labels.
svc_scores = svc_f.decision_function(F_test)

# same trick as before, take the percentile the holdout wanted rather than a
# fixed cut off, since a decision_function is not on any meaningful scale
best_t_svc, best_m_svc = tuned_macro(s_svc, yb)
pct_svc = (s_svc < best_t_svc).mean() * 100
pred_svc = (svc_scores >= np.percentile(svc_scores, pct_svc)).astype(int)

pd.DataFrame({"id": test_raw["id"].values, "label": pred_svc}).to_csv("Task3_Prediction_SVC.csv", index=False)
print("wrote Task3_Prediction_SVC.csv")
print("holdout macro f1 for the svm alone %.4f, against %.4f for the blend" % (best_m_svc, best_m))
print("predicted class 1 rate  svm %.4f  blend %.4f" % (pred_svc.mean(), pred.mean()))
print("the two files disagree on %d of %d documents" % ((pred_svc != pred).sum(), len(pred)))

wrote Task3_Prediction_SVC.csv
holdout macro f1 for the svm alone 0.8078, against 0.8413 for the blend
predicted class 1 rate  svm 0.5899  blend 0.6600
the two files disagree on 974 of 6999 documents


In [14]:
# Same checks as task 2 before uploading, now for both files. these are the
# things that would get a submission rejected or quietly scored wrong, and
# submissions are limited so it is worth the few seconds.
sample = pd.read_csv(DATA + "sample_submission.csv")
for fname in ["Task3_Prediction.csv", "Task3_Prediction_SVC.csv"]:
    check = pd.read_csv(fname)
    ok = (len(check) == 6999
          and check.columns.tolist() == ["id", "label"]
          and set(check["label"].unique()) <= {0, 1}
          and not check.isna().any().any()
          and list(check["id"].astype(str)) == list(sample["id"].astype(str)))
    print("%-28s rows %d  cols ok %s  labels ok %s  ids and order ok %s  ->  %s" % (
        fname, len(check),
        check.columns.tolist() == ["id", "label"],
        set(check["label"].unique()) <= {0, 1},
        list(check["id"].astype(str)) == list(sample["id"].astype(str)),
        "READY" if ok else "PROBLEM"))

Task3_Prediction.csv         rows 6999  cols ok True  labels ok True  ids and order ok True  ->  READY
Task3_Prediction_SVC.csv     rows 6999  cols ok True  labels ok True  ids and order ok True  ->  READY


In [15]:
# Summary of everything tried, for deliverable 3a.
#
# on the 5000 features we were given:
#   LogisticRegression C=1                                 0.7424
#   LogisticRegression C=10                                0.7378
#   LinearSVC C=1                                          0.7352
#   ComplementNB                                           0.6794
#   XGBoost n_estimators=400 max_depth=6 lr=0.1            0.7387
#   LightGBM n_estimators=400 num_leaves=63 lr=0.1         0.7473
#   our own logistic regression from task 1                0.7431
#   everything lands around 0.74, the features are the ceiling and not the model
#
# on my own features from the raw text, word 1-2gram 50k plus char_wb 3-5gram 100k:
#   LinearSVC C=1                                          0.8078
#   LightGBM n_estimators=500 num_leaves=63 lr=0.1         0.8389
#   50/50 blend of the two                                 0.8413
#
# what actually moved the number:
#   better features                          0.743 to 0.808   (+0.065)
#   boosting instead of a linear model       0.808 to 0.839   (+0.031)
#   blending the two models                  0.839 to 0.841   (+0.002)
#
# things I tried that did not make it into the final model:
#   pseudo labelling, came out flat on the holdout, but the holdout cannot
#     measure the thing it was meant to fix so it is untested rather than beaten
#   tuning C on the svm, 0.1 and 1 and 10 all land within 0.01 of each other
#   svm only, on the literature's recommendation. it holds up but loses by about
#     0.03, including under a simulated distribution shift
#
# the thing that is not really in the table:
#   adversarial validation says 71% of the test set is academic peer reviews and
#   is not the same distribution as anything we trained on, AUC 0.95. so every
#   number above is measured on the 29% that does look like training data, and
#   the real leaderboard score will come in lower. the local number is not wrong,
#   it is answering a narrower question than the one we are graded on.
#
# two files are written out, the blend and the svm on its own. submitting both
# is the only way to find out whether the literature was right that svms hold up
# better on data unlike the training set, since nothing local can measure it.
#
# what I would try next given more submissions:
#   spend one on the pseudo labelled version to see if it helps on the 71%
#   weight the char features higher, style should carry across genres better
#     than vocabulary does
#   proper k fold instead of one holdout before committing, boosting on 150000
#   features has real room to overfit and the private leaderboard is what counts